In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

#### ※ 集約関数 vs ウィンドウ関数  
- 集約関数の結果セットは、最終`SELECT`の結果セットとなる。  
- ウィンドウ関数の結果セットは、サブクエリの結果セットと同じであり、メインクエリと結合（集合演算、JOINなど）する。

## 1. ウィンドウ関数で一緒に使用するオプション

ウィンドウ関数は主に`OVER()`句と一緒に使用し、次の要素を組み合わせて計算範囲を指定する。

- `PARTITION BY`：データをグループに分ける。
- `ORDER BY`：グループ内の並べ替え基準を指定する。
- `ROWS / RANGE`：計算する行の範囲を指定する。

`ORDER BY`がない場合は行の順序がないため、`ROWS`や`RANGE`を使用できない。

- `ORDER BY`がない場合  
  → 計算範囲は`RANGE：最初 ～ 最後`と同じ

- `ORDER BY`があり、`ROWS / RANGE`句を指定しない場合  
  → `RANGE：最初 ～ 現在行`がデフォルト

- `ROWS`と`RANGE`は`ORDER BY`を基準に動作し、`PARTITION BY`がある場合は各グループ内で範囲を計算する。

#### ※ 基本構文

```sql
OVER (
    PARTITION BY ...
    ORDER BY ...
    ROWS BETWEEN ...
)
```

#### # ROWSとRANGEの違い

| `ROWS` | `RANGE` |
|---|---|
| `行の個数`を基準にする | `値の範囲`を基準にする |
| 同じ値でもそれぞれ別の行として扱う | 同じ値を1つの範囲として扱う |
| 最もよく使用される | 相対的に使用頻度が低い |
| 累積合計・移動平均などに適している | 順位・累積比率などに適している |

- `ROWS`は`行`を基準にするため、現在行までの行だけを計算対象にする。  
- 一方、`RANGE`は`値の範囲`を基準にするため、`ORDER BY`の値が同じ他の行も一緒に計算対象に含める。

次のデータがあるとする。

| id | score |
|---:|---:|
| 1 | 80 |
| 2 | 90 ← 現在行 |
| 3 | 90 |
| 4 | 100 |

### - 現在行が`id = 2`の場合：

#### 1) ROWSの場合

```sql
SUM(score) OVER(
    ORDER BY score
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

`行`を基準に最初から現在行まで計算する。

```text
80 + 90 = 170
```

#### 2) RANGEの場合

```sql
SUM(score) OVER(
    ORDER BY score
    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

`値`を基準にするため、現在行と同じ`score = 90`を持つ行もすべて含める。

```text
80 + 90 + 90 = 260
```

#### 結果比較

| id | score | ROWS | RANGE |
|---:|---:|---:|---:|
| 1 | 80 | 80 | 80 |
| 2 | 90 | 170 | 260 |
| 3 | 90 | 260 | 260 |
| 4 | 100 | 360 | 360 |

> `ROWS`：実際の行を基準に現在行まで計算する。  
> `RANGE`：`ORDER BY`の値を基準にし、同じ値を持つ行も同じ範囲として扱う。

そのため、重複する`ORDER BY`の値によって予想と異なる結果になることを避けるため、実務では`ROWS`を使用することが多い。

## 2. ウィンドウ範囲

`ROWS`では、現在行を基準にして前後の行を計算範囲として指定できる。

| ROWSの範囲 | 意味 | 使用例 |
|---|---|---|
| `UNBOUNDED PRECEDING ～ CURRENT ROW` | 最初の行から現在行まで | 累積合計、累積平均 |
| `1 PRECEDING ～ CURRENT ROW` | 前の1行 + 現在行 | 直近2行の合計 |
| `1 PRECEDING ～ 1 FOLLOWING` | 前の1行 + 現在行 + 次の1行 | 前後のデータ比較 |
| `CURRENT ROW ～ UNBOUNDED FOLLOWING` | 現在行から最後の行まで | 残りのデータの累積 |

#### 例)

学生の点数が次のようになっているとする。

| 学生 | 点数 |
|---|---:|
| A | 80 |
| B | 85 |
| C | 90 ← 現在行|
| D | 95 |
| E | 100 |

現在行が`C（90点）`の場合：

- `UNBOUNDED PRECEDING ～ CURRENT ROW`  
  → `A ～ C`：80, 85, 90

- `1 PRECEDING ～ CURRENT ROW`  
  → `B, C`：85, 90

- `1 PRECEDING ～ 1 FOLLOWING`  
  → `B, C, D`：85, 90, 95

- `CURRENT ROW ～ UNBOUNDED FOLLOWING`  
  → `C ～ E`：90, 95, 100

> `PRECEDING`：現在行より前  
> `CURRENT ROW`：現在行  
> `FOLLOWING`：現在行より後  
> `UNBOUNDED`：範囲の端まで

### OVER()の組み合わせを整理

```text
1) 関数() OVER()
→ 全体を1つのパーティションとして処理
→ 並べ替えなし
→ RANGE：最初 ～ 最後

2) 関数() OVER(PARTITION BY ...)
→ 複数のパーティションに分割
→ 並べ替えなし
→ RANGE：最初 ～ 最後

3) 関数() OVER(ORDER BY ...)
→ 1つのパーティション
→ 並べ替えあり
→ RANGE：最初 ～ 現在行

4) 関数() OVER(PARTITION BY ... ORDER BY ...)
→ 複数のパーティション
→ 各パーティション内で並べ替え
→ RANGE：最初 ～ 現在行
```

> `ORDER BY`がない場合、デフォルトの計算範囲は`最初 ～ 最後`となる。  
> `ORDER BY`がある場合、デフォルトは`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`となる。  
> `ROWS / RANGE`を明示すると、計算範囲を変更できる。


### RANGE・ROWS・全体合計の比較

```sql
SELECT
    title,
    length,

    SUM(length) OVER(
        ORDER BY length DESC 
    ) AS 値範囲累積,

    SUM(length) OVER(
        ORDER BY length DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS 行範囲累積,

    SUM(length) OVER(
        ORDER BY length DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS 全体合計

FROM film;
```

#### 1) 上映時間の累積合計

- 上映時間が短い映画から現在の映画までの`length`の合計を求める。

In [3]:
%%sql cumulative_length_result <<

SELECT
    title,
    length,
    SUM(length) OVER(
        ORDER BY length
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_length
FROM film;

[📊 全件の結果を見る](../../sql_study/results/7.23/cumulative_length_result.csv)

#### 2) 直近3本の映画の平均上映時間：前2行 + 現在行

- 現在の映画と、その直前2本を含む`3本`の平均上映時間を求める。

In [7]:
%%sql avg_length_3rows_result <<

SELECT
    title,
    length,
    AVG(length) OVER(
        ORDER BY length
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS avg_length
FROM film;

[📊 全件の結果を見る](../../sql_study/results/7.23/avg_length_3rows_result.csv)

#### 3) 直近3本の映画の平均上映時間：前1行 + 現在行 + 次1行

- 現在の映画を中心に、前後1本ずつを含む`3本`の平均上映時間を求める。

In [9]:
%%sql neighbor_avg_result <<

SELECT
    title,
    length,
    AVG(length) OVER(
        ORDER BY length
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ) AS neighbor_avg
FROM film;

[📊 全件の結果を見る](../../sql_study/results/7.23/neighbor_avg_result.csv)

#### 4) 現在行から最後まで（残りデータの累積）

- 現在の映画から最後の映画までの上映時間の合計を求める。

In [12]:
%%sql remaining_length_result <<

SELECT
    title,
    length,
    SUM(length) OVER(
        ORDER BY length
        ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
    ) AS remaining_length
FROM film;

[📊 全件の結果を見る](../../sql_study/results/7.23/remaining_length_result.csv)

## 3. PARTITION BY + ORDER BY + ROWSをすべて使用する例

`staff`ごとの日別売上を求めた後、直近`7日間（1週間）`の累積売上を計算し、7日間の累積売上が多いデータを検索する。

In [24]:
%%sql staff_daily_sales_result <<

SELECT
    p.staff_id,
    DATE(p.payment_date) AS pay_date,
    SUM(p.amount) AS daily_amount
FROM payment p
GROUP BY
    p.staff_id,
    DATE(p.payment_date)
ORDER BY
    p.staff_id,
    pay_date;

[📊 全件の結果を見る](../../sql_study/results/7.23/staff_daily_sales_result.csv)